In [1]:
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

# Lab 06: Exact Inference

In this session, we explore the sum-product and the variable elimination algorithms for exact inference. The goal of the first part is to fully understand the intuition behind the sum-product algorithm. We propose implementations for both algorithms.

_Note: As usual, questions marked with a (*) can be presented in class in exchange for up to 1 bonus point._

## Part 1: The Sum-Product algorithm

In this part, we consider an undirected tree. In the following question, we show that the case of directed trees also falls into this assumption. 

#### Question 1 (*)

Let $p(\mathbf{x})$ a distribution which factorizes over a directed tree. Show that there exists an undirected tree over which $p(\mathbf{x})$ factorizes. 


### Understanding the algorithm

We remind that the sum-product algorithm is based on the idea of passing messages from one node to the other, in order to compute some marginal distribution $p(x_i)$. The order of the nodes is determined by the structure of the tree, where node $X_i$ is taken as the root. 

We remind that the messages are defined by:
$$
\mu_{j \rightarrow i}(x_i) = \sum_{x_j} \psi_{ij}(x_i, x_j) \psi_j(x_j) \prod_{k \in \mathcal{C}(j)} \mu_{k \rightarrow j}(x_j)
$$
where $\mathcal{C}(j)$ are the children of node $X_j$ and with the convention that $\prod_{k \in \emptyset} = 1$.

#### Question 2 (*)

Consider a chain of size 3, with edges only $(1,2)$ and $(2,3)$.

a) Show that $p(x_1) = \frac{1}{Z} \mu_{2 \rightarrow 1}(x_1)$.<br>
b) For this same chain, check that $p(x_2) = \frac{1}{Z} \mu_{3 \rightarrow 2}(x_2) \mu_{1 \rightarrow 2}(x_2)$.

The sum-product algorithm works by computing messages from the leaves to the root. The final messages are used to compute:
$$
p(x_1) = \frac{1}{Z} \psi_i(x_i) \prod_{j \in \mathcal{C}(i)} \mu_{j \rightarrow i}(x_i)
$$


### Implementation of the algorithm

We first need to represent the undirected tree. For this, we separately represent the edges, in the form of a binary matrix, and the unary and binary potentials. 

Unary potentials are stored in a dictionary of a arrays, such that ```psi_u[i][k]```represents $\psi_i(k)$ (assuming that $X_i$ takes values in $\lbrace 1, \dotsc, K_i \rbrace$.  

Unary potentials are stored in a dictionary of arrays, such that ```psi_b[i,j][k,l]```represents $\psi_{ij}(k,l)$.

**Remark:** Note that in the code we define $\psi_{ij}$ and $\psi_{ji}$ while, mathematically speaking, the order of the nodes in the clique does not matter (mathematically, we have $\psi_{ij}(x_i, x_j) = \psi_{ji}(x_i, x_j)$). We make this choice because we want that ```psi_b[i,j][k,l]``` corresponds to the evaluation of $\psi_{ij}$ for $x_i = k$ and $x_j = l$, and ```psi_b[j,i][k,l]``` corresponds to the evaluation of $\psi_{ij}$ for $x_j = k$ and $x_i = l$. This implies that ```psi_b[j,i]``` is the transpose of ```psi_b[i,j]```.

In [14]:
edges = np.array([ [0, 1, 0, 0, 0], [1, 0, 1, 1, 0], [0, 1, 0, 0, 0], [0, 1, 0, 0, 1], [0, 0, 0, 1, 0] ])

psi_u = dict()
psi_u[0] = np.array([.6, .4])
psi_u[1] = np.array([1., 1.])
psi_u[2] = np.array([1., 1.])
psi_u[3] = np.array([1., 1.])
psi_u[4] = np.array([1., 1.])


psi_b = dict()

# p(x_1 = 0 | x_0 = 0) = 0.9
# p(x_1 = 0 | x_0 = 1) = 0.2

psi_b[0, 1] = np.array([[.9, .1], [.2, .8]])
psi_b[1, 0] = np.array([[.9, .2], [.1, .8]])

# p(x_2 = 0 | x_1 = 0) = 0.6
# p(x_2 = 0 | x_1 = 1) = 0.5

psi_b[2, 1] = np.array([[.6, .5], [.4, .5]])
psi_b[1, 2] = np.array([[.6, .4], [.5, .5]])


# p(x_3 = 0 | x_1 = 0) = 0.3
# p(x_3 = 0 | x_1 = 1) = 0.6

psi_b[3, 1] = np.array([[.3, .6], [.7, .4]])
psi_b[1, 3] = np.array([[.3, .7], [.6, .4]])

# p(x_4 = 0 | x_3 = 0) = 0.75
# p(x_4 = 0 | x_3 = 1) = 0.3

psi_b[4, 3] = np.array([[.75, .3], [.25, .7]])
psi_b[3, 4] = np.array([[.75, .25], [.3, .7]])


print(psi_u)
print(psi_b)

{0: array([0.6, 0.4]), 1: array([1., 1.]), 2: array([1., 1.]), 3: array([1., 1.]), 4: array([1., 1.])}
{(0, 1): array([[0.9, 0.1],
       [0.2, 0.8]]), (1, 0): array([[0.9, 0.2],
       [0.1, 0.8]]), (2, 1): array([[0.6, 0.5],
       [0.4, 0.5]]), (1, 2): array([[0.6, 0.4],
       [0.5, 0.5]]), (3, 1): array([[0.3, 0.6],
       [0.7, 0.4]]), (1, 3): array([[0.3, 0.7],
       [0.6, 0.4]]), (4, 3): array([[0.75, 0.3 ],
       [0.25, 0.7 ]]), (3, 4): array([[0.75, 0.25],
       [0.3 , 0.7 ]])}


Note that we don't define the potentials for edges that don't exist. 


#### Question 3 (*)

Manually compute $p(x_2)$ for this defined network.

<hr>

We will now implement the Sum-Product algorithm step by step. The first step of the algorithm is to compute the messages recursively. 

#### Question 4 (*)

a) Implement a method computing the message. For this, we can use the (already implemented) method ```get_children(node, parent, edges)``` which finds the children of a node, knowing its parent (note that the parent is unique, since we are in a tree).<br>
b) Implement the sum-product algorithm, computing the probability $p(x_i)$ for a given node $i$. 

In [ ]:
def get_children(node: int, parent: int, edges: np.ndarray) -> set[int]:
    neighbors = set(np.nonzero(edges[node,:])[0])
    if parent in neighbors: neighbors.remove(parent)
    return neighbors

def message(j: int, i: int, edges: np.ndarray) -> np.ndarray:
    combined = psi_u[j].copy()
    children = get_children(j, i, edges)
    if len(children) == 0:
        mu = combined @ psi_b[(j, i)]
        # mu /= mu.sum()
        return mu
    for k in children:
        combined *= message(k, j, edges)
    mu = combined @ psi_b[(j, i)]
    # mu /= mu.sum()
    return mu

In [20]:
message(2, 1, edges)

array([0.5, 0.5])

In [21]:
def sum_product(i: int, edges: np.ndarray) -> np.ndarray:
    proba = psi_u[i].copy()
    for j in get_children(i, -1, edges):
        proba *= message(j, i, edges)
    return proba / np.sum(proba)

In [22]:
sum_product(2, edges)

array([0.562, 0.438])

## Part 2: Variable elimination

We will now propose an implementation of the variable elimination algorithm. 

Unlike for the sum-product algorithm, we now need to track the potentials, and consequently we will not duplicate the potentials $\psi_{ij}$ and $\psi_{ji}$. We will take the convention that we take the potentials with the variables ordered from the smallest to the larget. The function to order the indices is ```order_indices(I)```. 

In [7]:
def order_indices(I):
    return tuple(sorted(I))

We start by instantiating the graph. 

In [8]:
edges = np.array([ [0, 1, 1, 0, 0], [1, 0, 0, 1, 0], [1, 0, 0, 1, 0], [0, 1, 1, 0, 1], [0, 0, 0, 1, 0] ])

psi = dict()
psi[0, 1] = psi_b[0, 1] = np.array([[.9, .1], [.2, .8]])
psi[1, 3] = np.array([[.3, .7], [.6, .4]])
psi[0, 2] = np.array([[.6, .4], [.5, .5]])
psi[2, 3] = np.array([[.75, .3], [.25, .7]])
psi[3, 4] = np.array([[.9, .05], [1.2, .3]])

In the initialization phase, we first store all the potentials into the list of active potentials.

In [9]:
def add_to_active_potentials(active_potentials, indices, potential):
    if indices in active_potentials:
        active_potentials[indices].append(potential)
    else:
        active_potentials[indices] = [potential]


def remove_from_active_potentials(active_potentials, indices):
    if indices in active_potentials:
        active_potentials.pop(indices)

        
active_potentials = dict()
for indices in psi:
    add_to_active_potentials(active_potentials, indices, psi[indices])

print(active_potentials)

{(0, 1): [array([[0.9, 0.1],
       [0.2, 0.8]])], (1, 3): [array([[0.3, 0.7],
       [0.6, 0.4]])], (0, 2): [array([[0.6, 0.4],
       [0.5, 0.5]])], (2, 3): [array([[0.75, 0.3 ],
       [0.25, 0.7 ]])], (3, 4): [array([[0.9 , 0.05],
       [1.2 , 0.3 ]])]}


We then store the evidence as part of the active potentials. 

In [10]:
evidence = dict({1: 1, 
                 4: 0})

for v in evidence:
    value = evidence[v]
    add_to_active_potentials(active_potentials, (v,), np.array([1-value, value]))

print(active_potentials)

{(0, 1): [array([[0.9, 0.1],
       [0.2, 0.8]])], (1, 3): [array([[0.3, 0.7],
       [0.6, 0.4]])], (0, 2): [array([[0.6, 0.4],
       [0.5, 0.5]])], (2, 3): [array([[0.75, 0.3 ],
       [0.25, 0.7 ]])], (3, 4): [array([[0.9 , 0.05],
       [1.2 , 0.3 ]])], (1,): [array([0, 1])], (4,): [array([1, 0])]}


We now need to implement the elimination operation itself (called update step, in the lecture). For this we will need to define the new potentials over the set of all referenced potentials. 

#### Question 5 (*)

Implement the update step. In the definition of the method, $n$ is the index of the variable to eliminate. 

In [23]:
def update(n, active_potentials):
    
    updated_potentials = deepcopy(active_potentials)
    
    # Find all potentials involving n
    potentials_of_interest = []
    involved_variables = set()
    
    for indices, potentials_list in list(active_potentials.items()):
        if n in indices:
            involved_variables.update(indices) # Add the variables to the set of involved variables
            for potential in potentials_list:
                potentials_of_interest.append((indices, potential))
            updated_potentials.pop(indices)
    
    involved_variables = order_indices(involved_variables)
    
    # Compute phi:
    
    phi_shape = tuple((2 for i in involved_variables))
    phi = np.ones(phi_shape)
    
    ### Your code here
    for indices, potential in potentials_of_interest:
        axes = [involved_variables.index(v) for v in indices]
        shape = [1]*len(involved_variables)
        for i,a in enumerate(axes):
            shape[a] = 2
        potential_reshaped = potential.reshape(shape)
        phi = phi * potential_reshaped

    
    # Compute tau_n:
    axis_to_sum = involved_variables.index(n)    
    involved_variables_minus_n = tuple(i for i in involved_variables if i != n)
    tau_n = np.sum(phi, axis=axis_to_sum)

    add_to_active_potentials(updated_potentials, involved_variables_minus_n, tau_n)
    return updated_potentials

#### Question 6 (*)

Use the update function to compute the probability $p(x_F | \bar{x}_E)$.

In [ ]:
def variable_elimination(F, evidence, psi, ordering):
    # Initialization
    active_potentials = dict()
    for indices in psi:
        add_to_active_potentials(active_potentials, indices, psi[indices])
    for v in evidence:
        value = evidence[v]
        add_to_active_potentials(active_potentials, (v,), np.array([1-value, value]))
    for var in ordering:
        if var not in evidence and var not in F:
            active_potentials = update(var, active_potentials)
    remaining_vars = set()
    for idx in active_potentials:
        remaining_vars.update(idx)
    remaining_vars = tuple(sorted(remaining_vars))
    final_shape = tuple(2 for _ in remaining_vars)
    final_phi = np.ones(final_shape)
    for indices, potential_list in active_potentials:
        axes = [remaining_vars.index(v) for v in indices]
        for potential in potential_list:
            shape = [1]*len(remaining_vars)
            for i,a in enumerate(axes):
                shape[a] = 2
            final_phi *= potential.reshape(shape)
    
    f = F[0]
    f_axis = remaining_vars.index(f)
    result = np.sum(final_phi, axis=tuple(i for i in range(len(remaining_vars)) if i!=f_axis))
    
    # normalize
    result = result / np.sum(result)
    return result

In [25]:
variable_elimination((0, 3), evidence, psi, [4, 2, 1, 0, 3])

Active:  {(0, 1): [array([[0.9, 0.1],
       [0.2, 0.8]])], (1, 3): [array([[0.3, 0.7],
       [0.6, 0.4]])], (3, 4): [array([[0.9 , 0.05],
       [1.2 , 0.3 ]])], (1,): [array([0, 1])], (4,): [array([1, 0])], (0, 3): [array([[0.55, 0.46],
       [0.5 , 0.5 ]])]}
